### 릿지(Ridge)
- 최소제곱 적합식의 수축 패널티라 불리는 항에 L2 패널티를 추가한것

- parameter
    - alpha 
        - 기본값 : 1.0
        - 규제의 강도 -> 클수록 회귀 계수가 작아지고 과대적합을 방지, 과소적합 위험 
    - solver
        - 기본값 : 'auto'
        - 데이터의 크기 / 희소성에 따라 적합한 solver를 선택 
        - 'cholesky' : 콜레스키 분해, 가장 기본적인 수식, 데이터의 개수가 작거나 중간 정도 
        - 'svd' : 특이값 분해, 다중공선성이 있는 경우에 조금 더 안정적으로 계산, 데이터의 개수가 작거나 중간 정도
        - 'sparse_cg', 'lspr' : 공액 기울기법(0인 데이터를 무시하고 계산), 대규모 데이터중 희소 데이터 
        - 'sag' : 확률적 평균 검사( 기존의 기울기만이 아니라 과거의 기울기의 평균을 이용하여 가중치 변화 ), 데이터의 행의 개수가 열의 개수보다 월등히 많은 경우 
        - 'saga' : sag 확장, 대규모 희소 데이터가 존재하는경우,  ElasticNet에서 사용
    - tol
        - 기본값 : 0.001 (1e-03)
        - 수렴의 판단 기준. 작을 수록 정밀. 속도면에서는 느려질수 있다. 
    - max_iter 
        - 기본값 : None
        - 최적화가 될때까지 최대 반복 횟수 
        - 데이터가 크거나 수치가 불안정한 경우에 필요
- 속성 
    - n_iter_
        - solver가 반복한 횟수를 입력 
    - coef_
        - 회귀 계수를 출력 (규제로 인한 가중치의 감소량때문에 선형회귀보다는 낮은 값들이 출력)

In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

In [ ]:
# 데이터셋 로드 
diabetes = load_diabetes()

In [ ]:
diabetes

In [ ]:
diabetes.frame

In [ ]:
df = pd.DataFrame(diabetes['data'], columns = diabetes['feature_names'])
df.head()

In [ ]:
# 알파 -> 규제의 강도 1e-03, 1e-02, 1e-01, 1e+00, 1e+01
# numpy에 logspace() 함수를 이용해서 데이터를 생성 
alphas = np.logspace(-3, 1, 5)
alphas

In [ ]:
# alpha 값에 따른 회귀 계수를 파악 
data = []

for a in alphas:
    # 모델 생성
    ridge = Ridge(alpha = a)
    # 모델에 학습 
    ridge.fit(diabetes['data'], diabetes['target'])
    # 학습이 된 모델에서 회귀계수를 data 빈 리스트에 추가 
    data.append(
        ridge.coef_
    )
data

In [ ]:
df_ridge = pd.DataFrame(
    data, index = alphas, columns = df.columns
)
df_ridge

In [ ]:
plt.semilogx(df_ridge)

plt.xticks(alphas, labels= np.log10(alphas))

plt.legend(labels = df_ridge.columns, bbox_to_anchor = (1, 1))
plt.xlabel(alphas)
plt.ylabel('coef')
plt.axhline(y = 0, color='black', linewidth = 3)

plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(diabetes['data'], diabetes['target'])

In [ ]:
plt.figure(figsize=(16, 15))
plt.axhline(y= 0, linestyle='--', linewidth = 3, color='black')

plt.plot(df_ridge.loc[0.001, ], '^-')
plt.plot(df_ridge.loc[0.01, ], 's')
plt.plot(df_ridge.loc[0.1, ], 'v')
plt.plot(df_ridge.loc[1.0, ], '*')
plt.plot(df_ridge.loc[10.0, ], 'o-')
plt.plot(lr.coef_, 'r', linewidth = 4, alpha = 0.5)

plt.legend( ['center', 0.001, 0.01, 0.1, 1, 10, 'linear'], bbox_to_anchor = (1, 1) )

plt.show()

- 연습
    - 당뇨병 데이터를 이용하여 위험 지수를 예측 
    - train, test는 8:2 비율로 나눠주고 
    - 선형회귀의 방법으로 mae의 값을 이용하여 성능을 확인 
    - 릿지에서 알파 값을 0.01, 0.1, 1인 경우로 모델을 생성하고 학습 뒤 예측하고 mae의 값을 확인 
    - 가장 성능이 좋은 모델은 무엇인가?

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    diabetes['data'], diabetes['target'], test_size=0.2, random_state=42
)

In [ ]:
lr = LinearRegression()
ridge_1 = Ridge(alpha=0.01)
ridge_2 = Ridge(alpha=0.1)
ridge_3 = Ridge(alpha=1)

In [ ]:
# 각각의 모델에 학습 
lr.fit(X_train, y_train)
ridge_1.fit(X_train, y_train)
ridge_2.fit(X_train, y_train)
ridge_3.fit(X_train, y_train)

In [ ]:
# 각각의 모델을 이용하여 예측
pred_lr = lr.predict(X_test)
pred_1 = ridge_1.predict(X_test)
pred_2 = ridge_2.predict(X_test)
pred_3 = ridge_3.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
print(round( mean_absolute_error(y_test, pred_lr) , 4))
print(round( mean_absolute_error(y_test, pred_1), 4 ))
print(round( mean_absolute_error(y_test, pred_2), 4 ) )
print(round( mean_absolute_error(y_test, pred_3), 4 ) )

In [ ]:
print(round( r2_score(y_test, pred_lr) , 4))
print(round( r2_score(y_test, pred_1), 4 ))
print(round( r2_score(y_test, pred_2), 4 ) )
print(round( r2_score(y_test, pred_3), 4 ) )

In [ ]:
# 가중치 * 입력값 + 절편 --> 예측값

# 가중치(회귀 계수)
w_list = ridge_3.coef_

w_list

In [ ]:
# 입력값 X_test
x_list = X_test[0]
x_list

In [ ]:
# 절편 값 
b = ridge_3.intercept_
b

In [ ]:
b_1 = b

for i in range(len(x_list)):
    b_1 += w_list[i] * x_list[i]

b_1

In [ ]:
pred_3[0]

In [ ]:
for w, x in zip(w_list, x_list):
    # print(w)
    # print(x)
    b += w * x
b

In [ ]:
list(zip(w_list, x_list))

### 라쏘(Lasso)
- 선형 회귀에서 L1 패널티를 추가한 방법

- parameter
    - alpha
        - 기본값 : 1.0 
        - 규제의 강도 
        - 규제가 강도가 너무 큰 경우에는 많은 컬럼의 회귀 계수가 0이 되어 모델이 단순화(과소적합)
        - 적절한 규제 강도를 사용시 회귀 계수가 0인 되는 컬럼들이 발생하면서 자동적으로 feature select(불필요한 컬럼 제거)
    - selection 
        - 기본값 : 'cyclic'
        - 좌표축 경사법의 업데이트의 순서 
            - 'cyclic' : 순차적으로 업데이트 
            - 'random' : 무작위 선택 업데이트 
    - percompute
        - 기본값 : 'auto'
        - 인자값은 bool
        - 그람행렬을 계산하고 메모리에 저장할것인가?
        - 피쳐의 개수가 너무 많은 경우라면 False을 사용하고 개수가 적다면 True
    - warm_start
        - 기본값 : False
        - 이전의 학습 결과를 기억 할것인가?

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
alpha = np.logspace(-3, 1, 5)
data = []

for a in alpha:
    lasso = Lasso(alpha=a)

    lasso.fit(diabetes['data'], diabetes['target'])
    data.append(lasso.coef_)

lasso_df = pd.DataFrame(data, index = alpha, columns = df.columns)
lasso_df

In [ ]:
plt.semilogx(lasso_df)

plt.show()

In [ ]:
plt.figure(figsize=(16, 15))
plt.axhline(y= 0, linestyle='--', linewidth = 3, color='black')

plt.plot(lasso_df.loc[0.001, ], '^-')
plt.plot(lasso_df.loc[0.01, ], 's')
plt.plot(lasso_df.loc[0.1, ], 'v')
plt.plot(lasso_df.loc[1.0, ], '*')
plt.plot(lasso_df.loc[10.0, ], 'o-')
plt.plot(lr.coef_, 'r', linewidth = 4, alpha = 0.5)

plt.legend( ['center', 0.001, 0.01, 0.1, 1, 10, 'linear'], bbox_to_anchor = (1, 1) )

plt.show()

In [ ]:
# 규제 강도에 따라서 mae, r2score를 확인 
x = diabetes['data']
y = diabetes['target']

In [ ]:
lasso_1 = Lasso(alpha=0.01)
lasso_2 = Lasso(alpha=0.1)
lasso_3 = Lasso(alpha=1)

In [ ]:
lasso_1.fit(X_train, y_train)
lasso_2.fit(X_train, y_train)
lasso_3.fit(X_train, y_train)

In [ ]:
pred_1 = lasso_1.predict(X_test)
pred_2 = lasso_2.predict(X_test)
pred_3 = lasso_3.predict(X_test)

In [ ]:
print( mean_absolute_error(y_test, pred_1) )
print( mean_absolute_error(y_test, pred_2) )
print( mean_absolute_error(y_test, pred_3) )

In [ ]:
print( r2_score(y_test, pred_1) )
print( r2_score(y_test, pred_2) )
print( r2_score(y_test, pred_3) )

In [ ]:
lasso_4 = Lasso(alpha=10)

lasso_4.fit(X_train, y_train)

lasso_4.predict(X_test)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# 다항 회귀 데이터셋 추가 
poly = PolynomialFeatures(degree=3)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

In [ ]:
X_train_poly.shape

In [ ]:
lasso_1.fit(X_train_poly, y_train)
lasso_2.fit(X_train_poly, y_train)
lasso_3.fit(X_train_poly, y_train)

In [ ]:
pred_1 = lasso_1.predict(X_test_poly)
pred_2 = lasso_2.predict(X_test_poly)
pred_3 = lasso_3.predict(X_test_poly)

In [ ]:
print( r2_score(y_test, pred_1) )
print( r2_score(y_test, pred_2) )
print( r2_score(y_test, pred_3) )

### 엘라스틱넷 (ElasticNet)

- 릿지 회귀, 라쏘 회귀를 절충한 알고리즘 
- L1, L2 패널티를 혼합하여 사용 -> 매개변수에서 혼합의 비율 

- parameter
    - alpha
        - 기본값 : 1.0
        - 규제의 강도 (L1, L2 모두 포함)
    - l1_ratio
        - 기본값 : 0.5
        - L1 패널티의 비중 
        - 0 : 릿지 회귀
        - 1 : 라쏘 회귀
        - 0에서 1 사잇값 : 엘라스틱넷

In [ ]:
from sklearn.linear_model import ElasticNet

In [ ]:
data = []

for a in alpha:
    ela = ElasticNet(alpha = a)

    ela.fit(x, y)
    data.append(ela.coef_)

ela_df = pd.DataFrame(data, index = alpha, columns = df.columns)

In [ ]:
ela_df

In [ ]:
plt.semilogx(ela_df)

plt.show()

In [ ]:
ela_1 = ElasticNet(alpha=0.01)
ela_2 = ElasticNet(alpha=0.1)
ela_3 = ElasticNet(alpha=1)

In [ ]:
ela_1.fit(X_train, y_train)
ela_2.fit(X_train, y_train)
ela_3.fit(X_train, y_train)

In [ ]:
pred_1 = ela_1.predict(X_test)
pred_2 = ela_2.predict(X_test)
pred_3 = ela_3.predict(X_test)

In [ ]:
print(r2_score(y_test, pred_1))
print(r2_score(y_test, pred_2))
print(r2_score(y_test, pred_3))

#### 연습 문제 
- 데이터 수집
    - sklearn 안에 샘플 데이터셋 중 fetch_cailfornia_housing 데이터셋을 로드 
    - 샘플 데이터를 이용하여 독립 변수, 종속 변수로 데이터를 각각 저장 
- 데이터 튜닝
    - 학습 데이터와 검증 데이터로 7:3의 비율로 나눠준다. 
- 선형 회귀 모델과, 엘라스틱넷을 모델을 생성 
    - alpha : 0.01, 0.1, 1
    - l1_ratio : 0, 0.5, 1
    - (반복문을 이용하여 성능 평가 | 모델의 성능 평가는 함수로 생성하고 반복문 )
- 각각의 모델들의 R2 Score를 확인 

In [ ]:
from sklearn.datasets import fetch_california_housing

In [ ]:
cali = fetch_california_housing()

In [ ]:
df = pd.DataFrame(cali['data'], columns = cali['feature_names'])

In [ ]:
df['price'] = cali['target']

In [73]:
df.head(1)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.02381,322.0,2.555556,37.88,-122.23,4.526


- 캘리포니아 부동산 정보 
    - MedInc : 중간 소득 
    - HouseAge : 주택 연식
    - AveRooms : 평균 방의 개수
    - AveBedrms : 평균 침실의 개수
    - Population : 인구( 해당 블록의 거주하는 인구의 수 )
    - AveOccup : 평균 거주자 수
    - Latitude : 위도
    - Longitude : 경도
    - price : 주택의 중간 가격

In [74]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [75]:
x = df.drop('price', axis=1).values
y = df['price'].values

In [77]:
# 7:3 비율로 train, test로 데이터를 분할 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42
)

In [ ]:
def models(x1, x2, y1, y2):
    # 모델들을 생성 
    lr = LinearRegression()
    # 엘라스틱에서 사용이 되는 변수의 목록 
    alpha_list = [0.01, 0.1, 1]
    l1_list = [0, 0.5, 1]
    model_names = ['Ridge', 'Elastic', 'Lasso']
    # 각 모델들의 성능을 저장하기 위해 빈 딕셔너리 생성 
    result = {}
    # 선형 모델 학습, 예측 r2score
    lr.fit(x1, y1)
    pred_lr = lr.predict(x2)
    r2_lr = r2_score(y2, pred_lr)

    result['Linear'] = r2_lr

    for model, l1 in zip(model_names, l1_list):
        for a in alpha_list:
            # 엘라스틱넷 모델을 생성
            ela = ElasticNet(alpha=a, l1_ratio=l1)

            ela.fit(x1, y1)

            pred_ela = ela.predict(x2)
            r2_ela = r2_score(y2, pred_ela)

            result[model+str(a)] = r2_ela
    return result

In [ ]:
res = models(X_train, X_test, y_train, y_test)

In [87]:
pd.Series(res).sort_values(ascending=False)

Elastic0.01    0.599895
Lasso0.01      0.599775
Ridge0.01      0.599435
Linear         0.595770
Ridge0.1       0.593151
Elastic0.1     0.575522
Lasso0.1       0.545118
Ridge1         0.525522
Elastic1       0.423795
Lasso1         0.288000
dtype: float64

In [88]:
# 극단치의 경계값을 구한다. (AveRooms)
q_1, q_3 = np.percentile( df['AveRooms'], [25, 75] )
print(q_1, q_3)

4.440716235896959 6.052380952380952


In [89]:
iqr = q_3 - q_1

upper_whis = q_3 + (1.5 * iqr)
lower_whis = q_1 - (1.5 * iqr)

print(upper_whis, lower_whis)

8.469878027106942 2.023219161170969


In [90]:
flag1 = df['AveRooms'] < lower_whis
flag2 = df['AveRooms'] > upper_whis

df.loc[flag1 | flag2, ]

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
73,0.4999,46.0,1.714286,0.571429,18.0,2.571429,37.81,-122.29,0.67500
155,8.8793,52.0,8.972868,1.131783,861.0,3.337209,37.81,-122.23,4.10300
511,13.4990,42.0,8.928358,1.000000,1018.0,3.038806,37.82,-122.22,5.00001
512,12.2138,52.0,9.210227,1.039773,1001.0,2.843750,37.82,-122.23,5.00001
514,12.3804,52.0,9.122715,1.033943,1192.0,3.112272,37.82,-122.23,5.00001
...,...,...,...,...,...,...,...,...,...
20408,7.7889,26.0,8.730038,1.045627,842.0,3.201521,34.19,-118.88,3.09900
20426,10.0472,11.0,9.890756,1.159664,415.0,3.487395,34.18,-118.69,5.00001
20428,8.7288,6.0,8.715842,1.102970,3385.0,3.351485,34.23,-118.83,4.25800
20436,12.5420,10.0,9.873315,1.102426,1179.0,3.177898,34.21,-118.69,5.00001


In [91]:
df.loc[flag2, ]

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
155,8.8793,52.0,8.972868,1.131783,861.0,3.337209,37.81,-122.23,4.10300
511,13.4990,42.0,8.928358,1.000000,1018.0,3.038806,37.82,-122.22,5.00001
512,12.2138,52.0,9.210227,1.039773,1001.0,2.843750,37.82,-122.23,5.00001
514,12.3804,52.0,9.122715,1.033943,1192.0,3.112272,37.82,-122.23,5.00001
517,8.7477,52.0,9.000000,1.134078,556.0,3.106145,37.82,-122.23,5.00001
...,...,...,...,...,...,...,...,...,...
20408,7.7889,26.0,8.730038,1.045627,842.0,3.201521,34.19,-118.88,3.09900
20426,10.0472,11.0,9.890756,1.159664,415.0,3.487395,34.18,-118.69,5.00001
20428,8.7288,6.0,8.715842,1.102970,3385.0,3.351485,34.23,-118.83,4.25800
20436,12.5420,10.0,9.873315,1.102426,1179.0,3.177898,34.21,-118.69,5.00001


In [92]:
df2 = df.loc[ ~( flag1 | flag2 ), ]
df2.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
count,20129.000000,20129.000000,20129.000000,20129.000000,20129.000000,20129.000000,20129.000000,20129.000000,20129.000000
mean,3.823653,28.813652,5.238801,1.062244,1437.994535,3.052913,35.623364,-119.581741,2.054875
std,1.791989,12.509678,1.154132,0.112499,1132.123106,9.909145,2.129506,2.000629,1.137022
min,0.499900,1.000000,2.032738,0.333333,3.000000,0.750000,32.540000,-124.350000,0.149990
25%,2.561200,18.000000,4.426656,1.005025,799.000000,2.432727,33.930000,-121.810000,1.194000
50%,3.525000,29.000000,5.197183,1.047170,1176.000000,2.821782,34.250000,-118.490000,1.795000
75%,4.714300,37.000000,5.983759,1.095960,1734.000000,3.286822,37.710000,-118.020000,2.631000
max,15.000100,52.000000,8.469738,3.099338,35682.000000,1243.333333,41.950000,-114.550000,5.000010


In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    df2.drop('price', axis=1), 
    df2['price'], 
    test_size=0.3, 
    random_state=42
)

In [94]:
res2 = models(X_train, X_test, y_train, y_test)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.572e+03, tolerance: 1.820e+00
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.922e+03, tolerance: 1.820e+00
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of th

In [95]:
pd.Series(res2).sort_values(ascending=False)

Linear         0.600531
Ridge0.01      0.595370
Elastic0.01    0.593559
Lasso0.01      0.589548
Ridge0.1       0.577523
Elastic0.1     0.555988
Lasso0.1       0.516866
Ridge1         0.499412
Elastic1       0.392867
Lasso1         0.243299
dtype: float64